<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

# Duckietown et ROS

Dans ce notebook, nous utiliserons ce que nous avons appris sur ROS dans les carnets précédents pour commencer à contrôler le Duckiebot, que ce soit dans l'environnement Duckiematrix ou sur du matériel réel.

## See what the Duckiebot Sees

Dans le terminal **sur votre ordinateur** (ni dans ce VSCode ni VNC), exécutez la commande suivante pour compiler votre code (nous écrirons le code plus bas, mais vous pouvez déjà le compiler et l'exécuter) :

    dts code build -R ROBOTNAME

La première exécution prendra un certain temps, car elle consiste à créer une image Docker, mais nous n'aurons besoin de le faire qu'une seule fois.

Ensuite, nous pouvons exécuter le code avec :

    dts code workbench -R ROBOTNAME [-m]

Où `ROBOTNAME` peut être le nom d'un Duckiebot virtuel ou réel. Si vous utilisez un robot virtuel fonctionnant avec Duckiematrix, vous devez inclure l'option `-m` (Duckiematrix doit être déjà en cours d'exécution : `dts code start_matrix`). Cette commande synchronise le code avec votre Duckiebot et vous demandera donc le mot de passe `ssh`, qui est `quackquack` si vous avez utilisé la valeur par défaut.

Vous pouvez exécuter le VNC Desktop (toujours dans un nouveau terminal) avec


    dts code vnc -R ROBOTNAME

Une fois le navigateur VNC ouvert, vous devriez voir les mêmes icônes que pour les précédents notebooks de cette série LX, sur le côté gauche. Double-cliquez sur l'icône intitulée `RQT Image...` ; cela ouvrira l'utilitaire ROS [rqt_image_view](https://wiki.ros.org/rqt_image_view).

Dans la barre de défilement située en haut à gauche de la fenêtre `rqt_image_view`, vous ne devriez voir qu'une seule option : `/ROBOTNAME/camera_node/image/compressed`. En la sélectionnant, vous verrez l'image capturée par la caméra du Duckiebot.
![rqt_image_view](../assets/rqt_image_view_duckiematrix.png)

## Terminals sommaire

Vous devriez avoir *3* terminaux ouverts à ce stade (*4* si vous utilisez Duckiematrix) :

1. L'editeur VSCode (`dts code editor`)
2. Le code qui execute (`dts code workbench`)
3. Le VNC (`dts code vnc`)
4. (optionnel) Le Duckiematrix (`dts code start_matrix`)

## Essayez le joystick.

Vous pouvez également ouvrir le joystick en double-cliquant sur l'icône « Joystick ». Essayez de cliquer sur les directions ou d'utiliser les touches fléchées de votre clavier. Vous devriez voir la direction correspondante sur le joystick s'allumer en vert, mais vous ne verrez pas le robot bouger dans la simulation (en supposant que la fenêtre `rqt_image_view` soit toujours ouverte depuis l'étape précédente).

Cependant, le joystick fait bien quelque chose. Pour voir ce qu'il fait, ouvrez un terminal en cliquant sur l'icône « LXTerminal » comme précédemment. Exécutez la commande suivante dans le terminal :

    rostopic echo /ROBOTNAME/joy

Cliquez ensuite à nouveau sur la fenêtre du joystick et commencez à cliquer sur les différentes directions, en observant attentivement les informations qui s'affichent dans le terminal. Portez une attention particulière à la section `axes` des données générées. Vous devriez voir l'une des valeurs changer (la position peut varier et la valeur peut être soit `+1.0`, soit `-1.0`, selon la direction dans laquelle vous appuyez).

![joy_echo](../assets/joy_echo.png)

Notez attentivement quel emplacement des données `axes` change de valeur lorsque vous actionnez le joystick dans chacune des quatre directions. Nous aurons besoin de ces informations ultérieurement.

Enfin, effectuez les opérations suivantes dans le terminal :

    rostopic list

pour voir tous les topics. Ce sera une longue liste, mais les deux qui nous intéressent ici sont :

```
/ROBOTNAME/joy [sensor_msgs/Joy]
/ROBOTNAME/wheels_driver_node/wheels_cmd [duckietown_msgs/WheelsCmdStamped]
```

Nous pouvons visualiser le sujet `/ROBOTNAME/joy` et nous savons au moins comment ces données sont published. Ce message est de type [Joy.msg](https://docs.ros.org/en/noetic/api/sensor_msgs/html/msg/Joy.html) (il s'agit d'un type de données standard fourni par ROS). Le topic sur lequel nous devons publish des messages pour faire bouger le robot est `/ROBOTNAME/wheels_driver_node/wheels_cmd`. Ce message est de type [WheelsCmdStamped.msg](https://github.com/duckietown/dt-ros-commons/blob/daffy/packages/duckietown_msgs/msg/WheelsCmdStamped.msg) (il s'agit d'un message que nous avons défini et qui est disponible car il est défini dans une image Docker en amont).

### Votre tâche

Notre objectif pour le reste de cet exercice sera de créer un ROS node qui recevra les messages published sur le topic `/ROBOTNAME/joy` et les utilisera pour publish des informations sur le topic `/ROBOTNAME/wheels_driver_node/wheels_cmd`, afin que le robot se déplace et que nous puissions le contrôler à l'aide du joystick.

## Écrire notre ROS node

Utilisez le menu **EXPLORER** situé à gauche de ce notebook pour trouver le fichier `packages` -> `src/dt-joystick-demo` -> `src` -> `dt-joystick-demo-node.py`. Cliquez sur le fichier `joystick-demo-node.py` pour l'ouvrir (vous pouvez éventuellement diviser l'écran à l'aide du bouton en haut à droite). Pour l'instant, il est vide.

Nous allons commencer par les bases. Tout d'abord, vous devez ajouter le code suivant en haut du fichier :

```python
#!/usr/bin/env python3
```

Cela indique à l'interpréteur Python qu'il s'agit d'un fichier Python.

Ensuite, nous devrons importer certains packages Python, `rospy`, `os`, ainsi que les types de messages que nous utiliserons ultérieurement.

```python
import rospy
import os
from sensor_msgs.msg import Joy
from duckietown_msgs.msg import WheelsCmdStamped
```

Tout en bas de votre fichier, créons la fonction principale qui sera appelée lors de l'exécution de votre script Python. Elle devrait ressembler à ceci :


```python
if __name__ == "__main__":
    # Initialize the node
    node = DTJoystickDemoNode()
    rospy.init_node('dt-joystick-demo-node')
    # Keep it spinning
    rospy.spin()
```

Ce code initialise une classe de type `DTJoystickDemoNode` (que nous n'avons pas encore définie), puis effectue des opérations spécifiques à ROS, notamment l'initialisation du node avec `rospy.init_node`, qui enregistre le node avec un nom unique, et enfin démarre l'exécution du node avec la fonction `rospy.spin()`. Cela garantit la réception des données des topics auxquels nous sommes subscribe et l'envoi effectif des données que nous publish (entre autres).

Nous sommes maintenant prêts à définir la classe `DTJoystickDemoNode`. Nous pouvons commencer par ce code :


```python
class DTJoystickDemoNode():
    def __init__(self):
```

### Construction du publisher et subscriber

L'étape principale à réaliser dans la fonction `__init__` est de configurer le publisher et le subscriber. Pour ce faire, nous aurons besoin de connaître le nom de notre robot. Ce nom est stocké dans une variable d'environnement, que nous pouvons récupérer grâce à la commande suivante :

```python
        veh_name = os.environ["VEHICLE_NAME"]
```
 

Maintenant, construisons le subscriber. Il devrait ressembler à quelque chose comme ceci :


```python
        self.sub_joy = rospy.Subscriber(
            "f/{veh_name}/joy", 
            Joy,
            self.process_joy
        )
```

**Remarque** : Comme pour tout code Python, veillez à respecter correctement l'indentation.

Cela initialise un objet `Subscriber`, lui indique d'écouter le topic `{veh_name}/joy`, précise que les données reçues sur ce topic doivent être de type `Joy`, et que nous les traiterons dans une fonction appelée `self.process_joy` (il s'agit d'une "function callback").

Le publisher ressemble a:

```python
        self.pub_wheel_cmds = rospy.Publisher(
            "f/{veh_name}/wheels_driver_node/wheels_cmd",
            WheelsCmdStamped
        )
```

Cela initialise un objet `Publisher` qui va publish des données de type `WheelsCmdStamped` sur un topics appelé `/{veh_name}/wheels_driver_node/wheels_cmd`.


### Traitement des données entrantes et calcul des commandes des roues.

La tâche principale que nous devons accomplir est de créer une fonction appelée `process_joy` qui traitera les messages entrants. Commencez par définir cette fonction :

```python
    def process_joy(self, msg):
```

Notez qu'en plus de la variable `self` habituelle pour une fonction définie dans une classe, il y a également la variable `msg`. C'est cette variable qui contiendra les données entrantes, en l'occurrence les commandes du joystick de type `Joy`.

Pour rendre le code aussi clair que possible, commençons par définir et initialiser la variable que nous allons finalement publish

```python
        cmd_to_publish = WheelsCmdStamped()
        cmd_to_publish.header = msg.header
        cmd_to_publish.vel_right = 0.0
        cmd_to_publish.vel_left = 0.0
```

La partie `header` de la variable contient des méta-informations, comme un timestamp. Il est recommandé de les copier à partir des données entrantes afin de pouvoir calculer des éléments tels que la latence, mais ne vous en souciez pas trop pour l'instant. Comme nous l'avons vu dans la définition du message [WheelsCmdStamped.msg](https://github.com/duckietown/dt-ros-commons/blob/daffy/packages/duckietown_msgs/msg/WheelsCmdStamped.msg), nous devons définir une valeur pour `vel_right` et une pour `vel_left` (pour chacune des deux roues du robot). Nous pouvons commencer par les initialiser à `0.0` afin que le robot ne bouge pas si aucun bouton de la joystick n'est activé.

Il nous suffit maintenant de vérifier quel bouton a été appuyé, puis d'attribuer les valeurs appropriées (positives ou négatives) aux roues correspondantes pour que le robot avance, recule ou tourne à gauche ou à droite. Comme nous l'avons vu lors de l'exercice précédent, ces informations sont transmises via les données `axes` du message [Joy.msg](https://docs.ros.org/en/noetic/api/sensor_msgs/html/msg/Joy.html). Plus précisément, vous devrez vérifier les valeurs des données `axes` et définir les valeurs de `cmd_to_publish.vel_right` et `cmd_to_publish.vel_left` de manière cohérente.



### Publishing 

Et enfin, la dernière chose à faire est de publish les données. Nous pouvons utiliser l'objet `Publisher` que nous avons défini précédemment :

```python
        self.pub_wheel_cmds.publish(cmd_to_publish)
```




### Vérification de la compilation du code

Dans le terminal **informatique** (celui où vous avez exécuté la commande `dts code workbench` – vous pouvez l'arrêter s'il est encore en cours d'exécution en appuyant sur `CTRL-C`), vous pouvez vérifier si votre code ne contient pas d'erreurs de syntaxe en exécutant la commande suivante :

    dts code build -R ROBOTNAME

Vous devriez voir plusieurs messages, mais la partie importante, si tout va bien, ressemblera à quelque chose du genre:

```bash
0.745 Starting >>> dt-joystick-demo                                                                                                                                  
0.745 Starting >>> duckietown                                                                                                                                        
0.745 Starting >>> duckietown_msgs                                                                                                                                   
0.746 Starting >>> duckietown_protocols                                                                                                                              
Finished <<< duckietown                          [ 0.2 seconds ]                                                                                                     
Finished <<< duckietown_protocols                [ 0.2 seconds ]                                                                                                     
Finished <<< duckietown_msgs                     [ 1.5 seconds ]                                                                                                     
Finished <<< dt-joystick-demo                    [ 2.3 seconds ]                                                                                                     
3.039 [build] Summary: All 4 packages succeeded!     
```

Si votre code contient une erreur de syntaxe, l'exécution échouera et un message d'erreur s'affichera.

## Tester votre code

Dans le terminal de votre ordinateur portable, vous êtes maintenant prêt à tester votre code. Comme précédemment, exécutez la commande suivante :

    dts code workbench -R ROBOTNAME [-m]

L'option `-m` est nécessaire si vous utilisez un robot virtuel dans Duckiematrix.

Ouvrez le navigateur VNC comme précédemment. Lancez `rqt_image_view` et le joystick, puis essayez d'appuyer sur les boutons du joystick. Si tout se passe bien, vous devriez pouvoir contrôler votre Duckiebot avec le joystick. Sinon, retournez au terminal où vous avez exécuté `dts code workbench` pour vérifier si des erreurs sont signalées par votre code. Si le problème persiste ou si le Duckiebot ne se comporte pas comme prévu, il y a probablement une erreur dans votre code et vous devrez procéder au débogage. Une bonne approche consiste à examiner les données published sur le topic `/ROBOTNAME/wheels_driver_node/wheels_cmd` à l'aide de `rostopic echo` ou `rqtplot`, comme nous l'avons vu précédemment.


Une fois que votre robot se comporte comme prévu, vous pouvez signaler au démonstrateur que vous avez terminé. Notez que même si vous avez effectué des développements dans Duckiematrix, la démonstration finale doit se faire sur le véritable Duckiebot.

Félicitations ! Vous avez réussi à écrire un node ROS pour envoyer des valeurs de commande depuis le joystick afin de faire bouger votre Duckiebot.

Vous remarquerez peut-être que notre implémentation présente certaines lacunes. Elle est très simple et de nombreuses améliorations pourraient y être apportées.